# mu-logsigma-encoder-head — ex2: two-head encoder (separate Linears for mu and logsigma) — equivalent to single-Linear+chunk

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `mu-logsigma-encoder-head`. Running the final beacon cell reports progress against the `VAE: mu+logsigma encoder head` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
import torch.nn as nn
import torch.nn.functional as F

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `VAE: mu+logsigma encoder head` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`mu-logsigma-encoder-head`** (exercise 2). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "mu-logsigma-encoder-head"
DD_SUBTOPIC = "VAE: mu+logsigma encoder head"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Encoder head — two-Linear variant equivalent to single-Linear+chunk

Ex1 used `Linear(d_in, 2*latent_dim)` then `.chunk(2, dim=-1)` to split
into `(mu, logsigma)`. The deepening move is the TWO-HEAD form:

```python
class TwoHeadEncoder(nn.Module):
    def __init__(self, d_in, latent_dim):
        super().__init__()
        self.fc_mu       = nn.Linear(d_in, latent_dim)
        self.fc_logsigma = nn.Linear(d_in, latent_dim)
    def forward(self, h):
        return self.fc_mu(h), self.fc_logsigma(h)
```

**Both forms are equivalent** when initialized correctly. Functionally
you can copy the single-Linear's weight matrix into the two heads:
row-block `[0:latent_dim]` → `fc_mu`, row-block `[latent_dim:]` →
`fc_logsigma`. Same forward.

**Why two heads in practice.** Lets you put `nn.Softplus` or
`Tanh()` only on the `logsigma` branch without touching `mu`.
Conditional/asymmetric initialization (e.g. small `fc_logsigma`
weight init for stable early KL) becomes trivial. Single-Linear forces
shared init on both halves.

**Parameter count is identical.** Two `Linear(d_in, k)` =
`Linear(d_in, 2k)` parameter-wise. No FLOPs difference.

### Exercise 2 — two-head encoder (separate Linears for mu and logsigma) — equivalent to single-Linear+chunk

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Bloom level: Apply
> LO: Apply a two-head encoder (`fc_mu`, `fc_logsigma`) and verify it is mathematically equivalent to a single `Linear(d_in, 2*latent)` + `chunk(2, dim=-1)` by copying row-block weights from the single-head form into the two-head module.
> Keywords: encoder, two-head, mu-logsigma, weight-copy
> ```

**KCs targeted:** `two-separate-linears-mu-logsigma`, `weight-copy-from-single-linear-row-blocks`

Implement `ex2_TwoHeadEncoder`, an `nn.Module`.

`__init__(d_in: int, latent_dim: int)` must:
1. `super().__init__()`.
2. `self.fc_mu       = nn.Linear(d_in, latent_dim)`.
3. `self.fc_logsigma = nn.Linear(d_in, latent_dim)`.

`forward(h: Tensor) -> tuple[Tensor, Tensor]` where `h: (B, d_in)`:
- Return `(self.fc_mu(h), self.fc_logsigma(h))` — both `(B, latent_dim)`.

Also implement `ex2_copy_from_single_head(two_head, single_linear)` which:
1. Takes a `two_head: ex2_TwoHeadEncoder` and a `single_linear: nn.Linear(d_in, 2 * latent_dim)`.
2. Copies row-block `[0:latent_dim]` of `single_linear.weight` into `two_head.fc_mu.weight` (and same slice of `bias` into `fc_mu.bias`).
3. Copies row-block `[latent_dim:]` of `single_linear.weight` into `two_head.fc_logsigma.weight` (and `bias`).
4. Uses `.data.copy_()` (in-place, no autograd tracking).
5. Returns nothing (mutation).

After copying, the two-head forward must produce the same `(mu, logsigma)` as `single_linear(h).chunk(2, dim=-1)` — that IS the equivalence test.

In [ ]:
class ex2_TwoHeadEncoder(nn.Module):
    def __init__(self, d_in: int, latent_dim: int):
        super().__init__()
        self.fc_mu       = nn.Linear(d_in, latent_dim)
        self.fc_logsigma = nn.Linear(d_in, latent_dim)

    def forward(self, h):
        return self.fc_mu(h), self.fc_logsigma(h)

def ex2_copy_from_single_head(two_head, single_linear):
    latent = two_head.fc_mu.out_features
    two_head.fc_mu.weight.data.copy_(single_linear.weight.data[:latent])
    two_head.fc_mu.bias.data.copy_(single_linear.bias.data[:latent])
    two_head.fc_logsigma.weight.data.copy_(single_linear.weight.data[latent:])
    two_head.fc_logsigma.bias.data.copy_(single_linear.bias.data[latent:])


<details><summary>Solution</summary>

```python
class ex2_TwoHeadEncoder(nn.Module):
    def __init__(self, d_in: int, latent_dim: int):
        super().__init__()
        self.fc_mu       = nn.Linear(d_in, latent_dim)
        self.fc_logsigma = nn.Linear(d_in, latent_dim)

    def forward(self, h):
        return self.fc_mu(h), self.fc_logsigma(h)

def ex2_copy_from_single_head(two_head, single_linear):
    latent = two_head.fc_mu.out_features
    two_head.fc_mu.weight.data.copy_(single_linear.weight.data[:latent])
    two_head.fc_mu.bias.data.copy_(single_linear.bias.data[:latent])
    two_head.fc_logsigma.weight.data.copy_(single_linear.weight.data[latent:])
    two_head.fc_logsigma.bias.data.copy_(single_linear.bias.data[latent:])
```

**The two forms are computationally equivalent.** A single `Linear(d_in, 2*latent)` is one matmul of shape `(B, d_in) @ (d_in, 2*latent)`. The two-head form is two matmuls of shape `(B, d_in) @ (d_in, latent)`. Same total FLOPs; same total params.

**Why `.data.copy_()` over assignment.** Reassigning `two_head.fc_mu.weight = nn.Parameter(...)` would replace the Parameter object — the optimizer's reference to the old Parameter would dangle. `.data.copy_()` mutates the storage in-place, preserving Parameter identity.

**Row-block slicing direction.** `nn.Linear.weight` has shape `(out_features, in_features)`. The first `latent` rows correspond to the first `latent` outputs (`y[:, :latent]`), which is what `chunk(2, dim=-1)[0]` returns. So `weight[:latent]` → `fc_mu`.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex2'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex2',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()